# Pythia-410M vs Mamba-370M: next-token predictions side by side

This is the first step toward the embedding-graft experiment in `init.md`. It loads both base models, builds `graft`
(a copy of Pythia with Mamba's input embedding swapped in, no adapter yet), and compares all three on the same
input tokens: next-token top-k and sampled continuations.

**Why this pair**

| | `EleutherAI/pythia-410m` | `state-spaces/mamba-370m-hf` |
|---|---|---|
| family | Transformer (GPT-NeoX) | SSM (Mamba) |
| layers | 24 | 48 |
| d_model | 1024 | 1024 |
| pretraining data | the Pile, 300B tokens | the Pile, 300B tokens |
| post-training | none (base) | none (base) |
| embedding rows | 50304 | 50280 |
| `W_U` tied to `W_E`? | no (`embed_in` / `embed_out`) | yes (`lm_head` = `backbone.embeddings`) |

- **They share a tokenizer.** The two `tokenizer.json` files are identical: GPT-NeoX-20B BPE, the same
  50,254-entry vocab, 50,009 merges and 25 added tokens, and the same pre/post-processing. This was checked
  offline during setup, so the notebook loads one shared tokenizer.
- **The real vocab is ids 0..50276 (`V = 50277`).** Embedding rows past that are padding, so logits are sliced
  to `[:V]` and both distributions cover the same tokens.
- `<|endoftext|>` (id 0) is BOS, EOS and pad for both. Encoding adds no BOS or other special tokens.

**Setup.** Run this once. The venv is shared by all projects in `ai_safety/`.
```bash
cd ~/dev/research/ai_safety
uv venv --python 3.12 .venv
uv pip install --python .venv/bin/python torch transformers ipykernel ipywidgets jupyterlab
.venv/bin/python -m ipykernel install --user --name ai_safety --display-name "Python (ai_safety .venv)"
```
Then select the kernel **"Python (ai_safety .venv)"**. The first load downloads about 2.4 GB of weights to
`~/.cache/huggingface`.

**Where to run.** A laptop CPU is enough: the three models in fp32 take about 4.8 GB of RAM. The notebook also
runs unchanged on a GPU, for example `l40-worker`, because `DEVICE` picks `cuda` when it's available.

**Expected Mamba warnings on CPU.** Mamba's first forward pass prints that `causal_conv1d_fn` and
`selective_scan_fn` are "falling back to its reference PyTorch implementation". Its first `generate()` prints the
same for `causal_conv1d_update` and `selective_state_update`. This happens because the CUDA-only packages
`causal_conv1d` and `mamba_ssm` aren't installed. The outputs are correct; it's just slower. On a GPU box,
installing those two packages gives the fast kernels.

## Config

In [ ]:
import math
import torch

MODELS = {
    "pythia": ("EleutherAI/pythia-410m", "step143000"),   # final pretraining checkpoint
    "mamba":  ("state-spaces/mamba-370m-hf", "main"),
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32   # the models are small; fp32 also keeps Mamba's scan exact on CPU
EOS_ID = 0              # <|endoftext|>: BOS, EOS and pad for both models
V = 50277               # real vocab size; embedding rows past this are padding

## Environment

In [ ]:
import sys
import transformers, tokenizers

print("python       ", sys.version.split()[0], sys.executable)
print("torch        ", torch.__version__)
print("transformers ", transformers.__version__)
print("tokenizers   ", tokenizers.__version__)
print("device       ", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## Load both models

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# One shared tokenizer: the two models' tokenizer.json files are identical (see intro).
repo, rev = MODELS["pythia"]
tok = AutoTokenizer.from_pretrained(repo, revision=rev)

models = {
    name: AutoModelForCausalLM.from_pretrained(repo, revision=rev, dtype=DTYPE).to(DEVICE).eval()
    for name, (repo, rev) in MODELS.items()
}
torch.set_grad_enabled(False)

print(f"{'model':8} {'class':22} {'params':>7} {'d_model':>7} {'layers':>6} {'emb rows':>8}  tied W_U = W_E")
for name, m in models.items():
    W_E, W_U = m.get_input_embeddings().weight, m.get_output_embeddings().weight
    n_params = sum(p.numel() for p in m.parameters())   # tied weights are counted once
    print(f"{name:8} {type(m).__name__:22} {n_params / 1e6:>6.0f}M {m.config.hidden_size:>7} "
          f"{m.config.num_hidden_layers:>6} {W_E.shape[0]:>8}  {W_E.data_ptr() == W_U.data_ptr()}")

## Graft: Mamba's input embedding into Pythia
`graft` is a copy of Pythia whose input embedding rows `[:V]` are replaced by Mamba's, times `EMB_SCALE`. Everything
else is still Pythia's: all 24 blocks, the final LayerNorm and the unembedding (`embed_out`). There's no adapter
yet, so in init.md terms the adapter is the identity. Rows from `V` up are padding that's never looked up, so they
keep Pythia's values. The original `pythia` is untouched.

The two embeddings differ in two ways:
- **Scale.** Mamba's rows are about 8.4× larger (mean row norm 6.68 vs 0.79). Setting `EMB_SCALE` to the printed
  `pythia/mamba` ratio (about 0.118) fixes the scale.
- **Basis.** The cosine similarity between a token's Pythia row and its Mamba row averages 0.0009, so the
  coordinates mean unrelated things. No scalar fixes that; that's the adapter's job.

With the raw swap (`EMB_SCALE = 1.0`), graft's next-token distributions are flat and dominated by punctuation
and newlines. Its greedy continuations are mostly newlines, and its samples are word salad.

After changing `EMB_SCALE`, re-run this cell. It rebuilds `graft` from a fresh copy of Pythia.

In [ ]:
import copy

EMB_SCALE = 1.0   # 1.0 = raw swap; the printed pythia/mamba ratio (~0.118) matches Pythia's mean row norm

pythia, mamba = models["pythia"], models["mamba"]
E_p = pythia.get_input_embeddings().weight   # [50304, 1024] gpt_neox.embed_in
E_m = mamba.get_input_embeddings().weight    # [50280, 1024] backbone.embeddings (tied to Mamba's lm_head)
P_norm, M_norm = E_p[:V].norm(dim=1).mean().item(), E_m[:V].norm(dim=1).mean().item()
print(f"mean row norm: pythia {P_norm:.3f}  mamba {M_norm:.3f}  pythia/mamba {P_norm / M_norm:.3f}")

graft = copy.deepcopy(pythia)
E_g = graft.get_input_embeddings().weight
E_g[:V] = EMB_SCALE * E_m[:V]
models["graft"] = graft

assert torch.equal(E_g[:V], EMB_SCALE * E_m[:V])
assert torch.equal(graft.get_output_embeddings().weight, pythia.get_output_embeddings().weight)
assert E_g.data_ptr() != E_p.data_ptr()   # the original pythia is untouched

## Next-token predictions, side by side
The prompt is encoded once, with nothing added, and the same token ids go to every model in `models`
(pythia, mamba, graft). Each model gets one
forward pass. The last-position logits are cast to float32, sliced to the real vocab `[:V]`, and log-softmaxed.
`prepend_eos=True` puts `<|endoftext|>` in front to mark the start of a document.

BPE attaches the space to the *next* word, so a trailing space in the prompt changes the input tokens. The
`input tokens` line shows exactly what the models see.

In [ ]:
def side_by_side(prompt, k=20, prepend_eos=False):
    ids = tok.encode(prompt, add_special_tokens=False)
    if prepend_eos:
        ids = [EOS_ID] + ids
    print("input tokens:", [tok.decode([i]) for i in ids])
    input_ids = torch.tensor([ids], device=DEVICE)

    tops = {}
    for name, m in models.items():
        logprobs = m(input_ids).logits[0, -1, :V].float().log_softmax(-1)
        tops[name] = torch.topk(logprobs, k)

    col = f"{'token':16} {'prob':>6} {'logprob':>8}"
    print(f"\n{'rank':>4}  " + "   |   ".join(f"{name:^{len(col)}}" for name in models))
    print(f"{'':>4}  " + "   |   ".join(col for _ in models))
    for r in range(k):
        cells = []
        for name in models:
            lp, i = tops[name].values[r].item(), tops[name].indices[r].item()
            cells.append(f"{tok.decode([i])!r:16.16} {math.exp(lp):>6.3f} {lp:>8.3f}")
        print(f"{r:>4}  " + "   |   ".join(cells))

side_by_side("The capital of France is")

## Sampling, side by side
Each model continues the same prompt twice. The first is greedy. The second is one seeded sample from the full
distribution (temperature 1, `top_k=0`, `top_p=1`). Every `generate()` argument is passed explicitly, because
otherwise transformers 5 samples with `top_k=50`.
- Padding ids (`V` and up) are suppressed, so they can't be emitted.
- `torch.manual_seed(seed)` runs right before each model's sample, so each column is reproducible no matter what
  the other models do.
- Generation stops early if a model emits `<|endoftext|>`.
- Newlines in the output are shown as `↵`, followed by a real line break.

In [ ]:
import itertools, textwrap

def wrap(text, width):
    # Word-wrap each line to `width`, keeping the text's own newlines (marked with ↵).
    lines = text.replace("\n", "↵\n").split("\n")
    return [row for line in lines for row in (textwrap.wrap(line, width) or [""])]

def print_columns(texts, width):
    cols = {name: wrap(t, width) for name, t in texts.items()}
    print(" | ".join(f"{name:<{width}}" for name in cols))
    print("-+-".join("-" * width for _ in cols))
    for row in itertools.zip_longest(*cols.values(), fillvalue=""):
        print(" | ".join(f"{cell:<{width}}" for cell in row))

def sample_side_by_side(prompt, max_new_tokens=40, seed=0, width=36):
    input_ids = torch.tensor([tok.encode(prompt, add_special_tokens=False)], device=DEVICE)
    greedy, sampled = {}, {}
    for name, m in models.items():
        common = dict(
            attention_mask=torch.ones_like(input_ids),
            max_new_tokens=max_new_tokens,
            eos_token_id=EOS_ID,                                   # stop at end-of-document
            pad_token_id=EOS_ID,
            suppress_tokens=list(range(V, m.config.vocab_size)),   # never emit padding ids
        )
        greedy[name] = tok.decode(m.generate(input_ids, do_sample=False, **common)[0])
        torch.manual_seed(seed)
        out = m.generate(input_ids, do_sample=True, temperature=1.0, top_k=0, top_p=1.0, **common)
        sampled[name] = tok.decode(out[0])

    print("greedy\n")
    print_columns(greedy, width)
    print(f"\nsample (seed {seed}, temperature 1, no top-k / top-p)\n")
    print_columns(sampled, width)

sample_side_by_side("The capital of France is")

## Scratch

In [ ]:
side_by_side("Once upon a time")
sample_side_by_side("Once upon a time")